# 💬 範例 11：顧客服務—顧客評論情緒分析與客訴分流 (NLP篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「客服主管、社群小編、品牌公關與顧客關係經理（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以客服公關白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、客服評論資料集介紹 (`reviews.csv`)

本數據集記錄了電商與門市跨渠道 1,500 筆真實顧客留言與情緒標籤：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 客服業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **評論編號** | 類別型 (`object`) | REV_10001 | 唯一流水編號 |
| **顧客評論內容** | 文字型 (`string`) | 「物流速度非常快，商品品質超棒！」 | 原始非結構化中文文字 |
| **評論星等** | 數值型 (`int64`) | 5 (星) | 客戶主觀星級評價 |
| **服務接觸渠道** | 類別型 (`object`) | Google商家、官網評價、蝦皮旗艦店、AppStore | 評論發佈渠道來源 |
| **字數長度** | 數值型 (`int64`) | 28 (字) | 評論長度 |
| **情緒標籤 (目標 y)** | 類別型 (`object`) | **「正面」 / 「負面」** | **預測目標：二元文本分類** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視星等與正負面情緒分佈。
3. **文字特徵關聯性與視覺化**：繪製字數長度與渠道之情緒交叉圖。
4. **TF-IDF 文字向量化與 Pipeline**：將中文詞彙轉換為數值權重矩陣。
5. **模型選型與 5 折交叉驗證**：評估邏輯回歸在文本分類上的穩定性。
6. **測試集大考與公關處遇**：評估混淆矩陣，輸出模型並示範新留言即時預警。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入顧客語意情緒分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析庫並確保中文字型可正常繪製文字分析長條圖。

---
💡 **AI 提問小抄：**

1. 📌 **非結構化文字資料之挑戰**
   > 「先前的範例都是規律的表格數字，而本範例是長短不一的中文句子。為什麼傳統演算法無法直接處理文字？什麼是文字特徵提取（Text Feature Extraction）？」


---  
### 【步驟 1】：載入資料與評論長度探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 reviews.csv 並檢視資料概況
df = pd.read_csv("reviews.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 正負面情緒分佈比例 ---")
print(df["情緒標籤"].value_counts(normalize=True))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  檢視 1,500 筆評論，確認正面約佔 66%、負面約佔 34%。

---
💡 **AI 提問小抄：**

1. 📌 **星等與情緒的一致性**
   > 「在真實評論中，有時顧客打了 5 顆星但留言卻寫『東西很爛根本不能用』（反諷或按錯）。純看星等 vs 用 NLP 分析文字內容，哪一個能更真實反映顧客心情？」


---  
### 【步驟 2】：星等評分與情緒標籤交叉長條圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製評論星等與發佈渠道的情緒分佈圖
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=df, x="評論星等", hue="情緒標籤", palette="coolwarm", ax=axes[0])
axes[0].set_title("各星等評分之正負面情緒分佈")
axes[0].set_xlabel("評論星等")
axes[0].set_ylabel("評論筆數")

sns.countplot(data=df, x="服務接觸渠道", hue="情緒標籤", palette="Set2", ax=axes[1])
axes[1].set_title("不同服務接觸渠道之情緒分佈")
axes[1].set_xlabel("接觸渠道")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  視覺化確認 1~2 星大多為負評、4~5 星為好評，並比較 Google 商家與官網的評價健康度。

---
💡 **AI 提問小抄：**

1. 📌 **負面評價的傳播效應**
   > 「Google 商家上的 1 星負評對實體門市的進店率有致命打擊。為什麼建立這套自動分類系統能幫助客服在第一時間攔截並妥善處理客訴？」


---  
### 【步驟 3】：TF-IDF 文字向量化與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立 TF-IDF 文本向量化管線
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df["顧客評論內容"]
y = (df["情緒標籤"] == "正面").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe_sentiment = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=500)),
    ("classifier", LogisticRegression(random_state=42))
])

print(f"✅ 資料分割完成！訓練集：{len(X_train)} 筆文本，測試集：{len(X_test)} 筆文本")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  利用 `TfidfVectorizer` 將每條評論轉換成 500 個關鍵字詞頻權重，並與邏輯回歸分類器組合成 Pipeline。

---
💡 **AI 提問小抄：**

1. 📌 **TF-IDF 的核心直覺解釋**
   > 「請用大白話解釋：TF（詞頻）與 IDF（逆向文件頻率）各自在算什麼？為什麼『包裝破損』這種少見但關鍵的詞會被賦予超高權重？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證評估文字分類表現
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe_sentiment, X_train, y_train, cv=5, scoring="roc_auc")
print(f"TF-IDF 邏輯回歸 5 折交叉驗證 平均 ROC-AUC: {scores.mean():.4f} (±{scores.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  運用 5 折交叉驗證客觀檢驗 TF-IDF 模型對未見過評論的泛化鑑別度。

---
💡 **AI 提問小抄：**

1. 📌 **邏輯回歸在文本分類的高效性**
   > 「為什麼在文字分類任務中，簡單快速的『TF-IDF + 邏輯回歸』往往表現得非常出色甚至不輸複雜的樹模型？高維稀疏矩陣的特性是什麼？」


---  
### 【步驟 5】：測試集大考與指標解讀 (混淆矩陣, Recall, ROC-AUC)

In [ ]:
# [步驟 5 程式碼] 在全新測試集進行評估並繪製混淆矩陣
from sklearn.metrics import classification_report, confusion_matrix

pipe_sentiment.fit(X_train, y_train)
y_pred = pipe_sentiment.predict(X_test)

print("=== 測試集最終語意分類報告 ===")
print(classification_report(y_test, y_pred, target_names=["負面(0)", "正面(1)"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=["預測負面", "預測正面"], yticklabels=["實際負面", "實際正面"])
plt.title("顧客評論情緒分析混淆矩陣")
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  評估 300 筆獨立測試評論的分類準確度與混淆矩陣熱力圖。

---
💡 **AI 提問小抄：**

1. 📌 **負評召回率 (Negative Recall)**
   > 「在客訴管理中，最害怕把負評誤判為正評。從混淆矩陣來看，我們的模型成功攔截了多少比例的真實負面評價？」


---  
### 【步驟 6】：新顧客評論即時推論 (Inference) 與公關回覆指引

In [ ]:
# [步驟 6 程式碼] 保存模型管線並示範新進評論即時分析
import joblib

joblib.dump(pipe_sentiment, "model.joblib")
print("✅ NLP 文字情緒分析管線已保存至 model.joblib！")

# 模擬剛剛在 Google 商家進來的一則新顧客留言
new_comment = ["出貨效率很高，贈品也很精緻，滿意的購物體驗！"]

loaded_model = joblib.load("model.joblib")
pos_prob = loaded_model.predict_proba(new_comment)[0][1]

print(f"\n🔔 評論內容：{new_comment[0]}")
print(f"預估正面情緒機率：{pos_prob*100:.1f}%")
if pos_prob >= 0.5:
    print("😊 判定為正向滿意評價！系統自動指派日常感謝回覆範本。")
else:
    print("🚨 判定為負面客訴！系統自動建立 P1 優先級客服工單並通報主管！")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  保存模型並展示新留言輸入即時推論情緒正負機率與自動派工策略。

---
💡 **AI 提問小抄：**

1. 📌 **自動化社群聆聽 (Social Listening)**
   > 「如何將此模型整合至 Python 爬蟲，每天自動監控 PTT、Dcard 與 Facebook 官方粉專上的品牌討論，只要偵測到負評飆高就即時發送 LINE 通報公關團隊？」
